In [20]:
import os
from getpass import getpass
from dotenv import load_dotenv

try:
    load_dotenv()
except Exception:
    pass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")

if not os.getenv("SERPAPI_API_KEY"):
    os.environ["SERPAPI_API_KEY"] = getpass("Enter SERPAPI_API_KEY: ")

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Enter PINECONE_API_KEY: ")

print("GROQ_API_KEY configured:", bool(os.getenv("GROQ_API_KEY")))
print("SERPAPI_API_KEY configured:", bool(os.getenv("SERPAPI_API_KEY")))
print("PINECONE_API_KEY configured:", bool(os.getenv("PINECONE_API_KEY")))

GROQ_API_KEY configured: True
SERPAPI_API_KEY configured: True
PINECONE_API_KEY configured: True


In [21]:
import os
os.environ.setdefault("USER_AGENT", "Mozilla/5.0 Agentic-RAG-Demo")

from langchain_community.document_loaders import WebBaseLoader

def load_web_pages(urls, user_agent="Mozilla/5.0 Agentic-RAG-Demo"):
    """
    Load one or more web pages and return a list of LangChain Documents.

    Args:
        urls: a single URL (str) or a list of URLs
        user_agent: User-Agent header sent with the request

    Returns:
        list of Document objects (one per URL)
    """
    # Allow passing a single string
    if isinstance(urls, str):
        urls = [urls]

    loader = WebBaseLoader(
        web_paths=urls,
        requests_kwargs={"headers": {"User-Agent": user_agent}},
    )

    try:
        docs = loader.load()
    except Exception as e:
        print(f"Error loading pages: {e}")
        return []

    return docs

docs1 = load_web_pages("https://docs.langchain.com/oss/python/langgraph/agentic-rag")
print(docs1)

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/agentic-rag', 'title': 'Build a custom RAG agent with LangGraph - Docs by LangChain', 'description': 'Build a custom retrieval agent with LangGraph that decides when to search a vector store or respond directly.', 'language': 'en'}, page_content='\n\n\n\n\n\nBuild a custom RAG agent with LangGraph - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep Re

In [22]:
import re
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

def load_pdf(
    file_path,
    extract_images=False,
    clean=True,
    min_chars=20,
    verbose=True,
):
    """
    Load a PDF using PyMuPDFLoader and return a list of LangChain Documents
    (one per page).

    Args:
        file_path (str | Path): path to the PDF file.
        extract_images (bool): run OCR on images inside the PDF
                               (needs: pip install rapidocr-onnxruntime).
        clean (bool): normalize whitespace in the extracted text.
        min_chars (int): pages with fewer characters than this are dropped.
        verbose (bool): print a summary of what was loaded.

    Returns:
        list[Document]: Documents with cleaned text and enriched metadata.
    """
    # 1. Validate the file
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"PDF not found: {path}")
    if path.suffix.lower() != ".pdf":
        raise ValueError(f"Not a PDF file: {path.name}")

    # 2. Load with PyMuPDFLoader
    try:
        loader = PyMuPDFLoader(str(path), extract_images=extract_images)
        docs = loader.load()
    except Exception as e:
        raise RuntimeError(f"Failed to load '{path.name}': {e}")

    total_pages = len(docs)
    cleaned_docs = []
    skipped_pages = []

    # 3. Clean, filter and enrich each page
    for doc in docs:
        text = doc.page_content

        if clean:
            text = re.sub(r"[ \t]+", " ", text)       # collapse spaces and tabs
            text = re.sub(r"\n{3,}", "\n\n", text)    # max one blank line
            text = text.strip()

        page_index = doc.metadata.get("page")

        # Skip blank or image-only pages
        if len(text) < min_chars:
            skipped_pages.append(page_index)
            continue

        doc.page_content = text
        doc.metadata["file_name"] = path.name
        if isinstance(page_index, int):
            doc.metadata["page_number"] = page_index + 1   # starts at 1

        cleaned_docs.append(doc)

    # 4. Summary and warnings
    if verbose:
        print(f"File          : {path.name}")
        print(f"Total pages   : {total_pages}")
        print(f"Pages kept    : {len(cleaned_docs)}")
        print(f"Pages skipped : {len(skipped_pages)}")

    if not cleaned_docs:
        print(
            "WARNING: no text was extracted. The PDF may be scanned (image-only). "
            "Try extract_images=True or an OCR-based approach."
        )

    return cleaned_docs

docs2 = load_pdf("C:\\Users\\mahat\\Downloads\\Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf")

print(len(docs2))                    # number of pages with text
print(docs2[0].page_content[:300])   # first 300 characters of page 1
print(docs2[0].metadata)


File          : Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf
Total pages   : 19
Pages kept    : 19
Pages skipped : 0
19
IEEE TRANSACTIONS ON ARTIFICIAL INTELLIGENCE, VOL. 6, NO. 12, DECEMBER 2025
3165
Generative Agents in Agent-Based Modeling:
Overview, Validation, and Emerging Challenges
Carlo Adornetto
, Adrian Mora
, Kai Hu
, Leticia Izquierdo Garcia
, Parfait Atchade-Adelomou
,
Gianluigi Greco
, Member, IEEE, Lui
{'producer': 'Adobe LiveCycle PDF Generator; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-23T23:20:44+05:30', 'source': 'C:\\Users\\mahat\\Downloads\\Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf', 'file_path': 'C:\\Users\\mahat\\Downloads\\Generative_Agents_in_Agent-Based_Modeling_Overview_Validation_and_Emerging_Challenges.pdf', 'total_pages': 19, 'format': 'PDF 1.7', 'title': 'Generative Agents

In [35]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(
    docs,
    chunk_size=1000,
    chunk_overlap=200,
    add_chunk_id=True,
    verbose=True,
):
    """
    Split LangChain Documents into smaller chunks.

    Args:
        docs (list[Document]): Documents from a loader (PDF, web, etc.).
        chunk_size (int): maximum characters per chunk.
        chunk_overlap (int): characters repeated between neighbouring chunks.
        add_chunk_id (bool): add a chunk number to each chunk's metadata.
        verbose (bool): print a summary.

    Returns:
        list[Document]: smaller Documents with the original metadata kept.
    """
    # 1. Validate input
    if not docs:
        print("WARNING: no documents to split.")
        return []
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size.")

    # 2. Create the splitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    # 3. Split (metadata is copied to every chunk)
    chunks = splitter.split_documents(docs)

    # 4. Add a chunk number to the metadata
    if add_chunk_id:
        for i, chunk in enumerate(chunks):
            chunk.metadata["chunk_id"] = i

    # 5. Summary
    if verbose:
        lengths = [len(c.page_content) for c in chunks]
        print(f"Input documents : {len(docs)}")
        print(f"Chunks created  : {len(chunks)}")
        print(f"Avg chunk size  : {sum(lengths) // len(lengths)} characters")
        print(f"Min / Max size  : {min(lengths)} / {max(lengths)}")

    return chunks

chunks1 = split_documents(docs1)
chunks2 = split_documents(docs2)
print(len(chunks1), len(chunks2))

Input documents : 1
Chunks created  : 29
Avg chunk size  : 738 characters
Min / Max size  : 59 / 990
Input documents : 19
Chunks created  : 148
Avg chunk size  : 936 characters
Min / Max size  : 264 / 999
29 148


In [36]:
from langchain_huggingface import HuggingFaceEmbeddings

def get_embedding_model(model_name="sentence-transformers/all-MiniLM-L6-v2"):
    """Create an embedding model that runs on your own computer."""
    return HuggingFaceEmbeddings(
        model_name=model_name,
        encode_kwargs={"normalize_embeddings": True},  # helps similarity search
    )


def embed_chunks(chunks, embedding_model, verbose=True):
    """
    Convert chunks (Documents) into embedding vectors.

    Args:
        chunks (list[Document]): chunks from split_documents().
        embedding_model: a LangChain embeddings object (created if None).
        verbose (bool): print a summary.

    Returns:
        tuple: (vectors, embedding_model)
            vectors = list of lists of floats, one per chunk
    """
    if not chunks:
        print("WARNING: no chunks to embed.")
        return [], embedding_model

    if embedding_model is None:
        embedding_model = get_embedding_model()

    # 1. Extract the text of each chunk
    texts = [chunk.page_content for chunk in chunks]

    # 2. Convert all texts to vectors
    vectors = embedding_model.embed_documents(texts)

    # 3. Summary
    if verbose:
        print(f"Chunks embedded   : {len(vectors)}")
        print(f"Vector dimensions : {len(vectors[0])}")

    return vectors

# Embedding vector
embedding_model = get_embedding_model()
vectors1 = embed_chunks(chunks1,embedding_model)
print(len(vectors1))        # same as number of chunks
print(len(vectors1[0]))     # 384 for all-MiniLM-L6-v2
print(vectors1[0][:5])      # first 5 numbers of the first vector

vectors2 = embed_chunks(chunks2,embedding_model)
print(len(vectors2))        # same as number of chunks
print(len(vectors2[0]))     # 384 for all-MiniLM-L6-v2
print(vectors2[0][:5])      # first 5 numbers of the first vector

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9950.56it/s]


Chunks embedded   : 29
Vector dimensions : 384
29
384
[-0.08519567549228668, 0.07107431441545486, 0.031059520319104195, -0.07544715702533722, -0.11793510615825653]
Chunks embedded   : 148
Vector dimensions : 384
148
384
[-0.05382286012172699, -0.06611452996730804, -0.01458174642175436, 0.04189278185367584, -0.04227723181247711]


In [37]:
import os
from dotenv import load_dotenv

load_dotenv()   # reads the .env file in your project folder

PINECONE_INDEX_NAME = "enterprise-ai-agents"
PINECONE_NAMESPACE = "Agent_AI"

if not os.getenv("PINECONE_API_KEY"):
    raise EnvironmentError("PINECONE_API_KEY is missing in your .env file.")

In [38]:
from pinecone import Pinecone

def store_vectors_in_pinecone(
    chunks,
    vectors,
    index_name="enterprise-ai-agents",
    namespace="Agent_AI",
    batch_size=100,
):
    if len(chunks) != len(vectors):
        raise ValueError("Chunks and vectors count must match.")

    pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
    index = pc.Index(index_name)

    records = []

    for i, (chunk, vector) in enumerate(zip(chunks, vectors)):
        metadata = {
            **chunk.metadata,
            "text": chunk.page_content,
        }

        records.append({
            "id": f"{chunk.metadata.get('file_name', 'document')}-{chunk.metadata.get('chunk_id', i)}",
            "values": vector,
            "metadata": metadata,
        })

    # Upload in batches
    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]

        index.upsert(
            vectors=batch,
            namespace=namespace,
        )

    print(f"Stored {len(records)} vectors in Pinecone")
    print(f"Namespace: {namespace}")

    return index

vectorstore = store_vectors_in_pinecone(
    chunks1,
    vectors1,
    index_name=PINECONE_INDEX_NAME,
    namespace=PINECONE_NAMESPACE,
)

Stored 29 vectors in Pinecone
Namespace: Agent_AI


In [39]:
vectorstore = store_vectors_in_pinecone(
    chunks2,
    vectors2,
    index_name=PINECONE_INDEX_NAME,
    namespace=PINECONE_NAMESPACE,
)

Stored 148 vectors in Pinecone
Namespace: Agent_AI


In [40]:
import os
from pinecone import Pinecone

def similarity_search(
    query,
    embedding_model,
    index_name=PINECONE_INDEX_NAME,
    namespace=PINECONE_NAMESPACE,
    top_k=3,
    metadata_filter=None,
):
    """
    Connect independently to Pinecone and retrieve relevant chunks.

    Args:
        query (str): User's search question.
        embedding_model: Embedding model used during ingestion.
        index_name (str): Pinecone index name.
        namespace (str): Namespace to search.
        top_k (int): Number of results to retrieve.
        score_threshold (float | None): Minimum similarity score.
        metadata_filter (dict | None): Pinecone metadata filter.
        verbose (bool): Print retrieval information.

    Returns:
        list[dict]: Retrieved documents with scores and metadata.
    """

    if not query or not query.strip():
        raise ValueError("Query cannot be empty.")

    api_key = os.environ.get("PINECONE_API_KEY")

    if not api_key:
        raise EnvironmentError("PINECONE_API_KEY is missing.")

    # 1. Create a NEW Pinecone client
    pc = Pinecone(api_key=api_key)

    # 2. Connect to the existing index
    index = pc.Index(index_name)

    # 3. Convert the query into an embedding vector
    query_vector = embedding_model.embed_query(query)

    # 4. Search Pinecone
    search_response = index.query(
        vector=query_vector,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True,
        filter=metadata_filter,
    )

    return search_response

ans = similarity_search("Give me the step of Preprocess documents?",embedding_model)

In [52]:
ans1 = similarity_search("What happens if retrieved documents are not relevant in Agentic RAG?",embedding_model)
print(ans1["matches"][2])

{'id': 'document-1',
 'metadata': {'chunk_id': 1.0,
              'description': 'Build a custom retrieval agent with LangGraph '
                             'that decides when to search a vector store or '
                             'respond directly.',
              'language': 'en',
              'source': 'https://docs.langchain.com/oss/python/langgraph/agentic-rag',
              'text': 'Documentation IndexFetch the complete documentation '
                      'index at: /llms.txtUse this file to discover all '
                      'available pages before exploring further.Skip to main '
                      'contentWould you notice if your agent got worse in '
                      'production? Tell us in our annual survey for the chance '
                      'to win LangChain swag.Docs by LangChain home '
                      'pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry '
                      'LangSmithSearch...NavigationLangGraphBuild a custom RAG '
           

In [59]:
print(ans1["matches"])

[{'id': 'document-0',
 'metadata': {'chunk_id': 0.0,
              'description': 'Build a custom retrieval agent with LangGraph '
                             'that decides when to search a vector store or '
                             'respond directly.',
              'language': 'en',
              'source': 'https://docs.langchain.com/oss/python/langgraph/agentic-rag',
              'text': 'Build a custom RAG agent with LangGraph - Docs by '
                      'LangChain',
              'title': 'Build a custom RAG agent with LangGraph - Docs by '
                       'LangChain'},
 'score': 0.482962817,
 'values': []}, {'id': 'document-2',
 'metadata': {'chunk_id': 2.0,
              'description': 'Build a custom retrieval agent with LangGraph '
                             'that decides when to search a vector store or '
                             'respond directly.',
              'language': 'en',
              'source': 'https://docs.langchain.com/oss/python/langgra

In [53]:
# Load Existing Index

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# Connect to Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Load existing Pinecone index
index = pc.Index(PINECONE_INDEX_NAME)

# Connect existing index with LangChain
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embedding_model,
    namespace=PINECONE_NAMESPACE,
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3,
        "namespace": PINECONE_NAMESPACE,
    }
)

print("Existing Pinecone index loaded:", PINECONE_NAMESPACE)

Existing Pinecone index loaded: Agent_AI


In [56]:
ans11 = retriever.invoke("What happens if retrieved documents are not relevant in Agentic RAG?")

In [57]:
print(ans11)

[Document(id='document-0', metadata={'chunk_id': 0.0, 'description': 'Build a custom retrieval agent with LangGraph that decides when to search a vector store or respond directly.', 'language': 'en', 'source': 'https://docs.langchain.com/oss/python/langgraph/agentic-rag', 'title': 'Build a custom RAG agent with LangGraph - Docs by LangChain'}, page_content='Build a custom RAG agent with LangGraph - Docs by LangChain'), Document(id='document-2', metadata={'chunk_id': 2.0, 'description': 'Build a custom retrieval agent with LangGraph that decides when to search a vector store or respond directly.', 'language': 'en', 'source': 'https://docs.langchain.com/oss/python/langgraph/agentic-rag', 'title': 'Build a custom RAG agent with LangGraph - Docs by LangChain'}, page_content='SQL assistantLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextHuman-in-the-loopGraph APIFunctional APIAdditional

In [ ]:
### Embedding process is complete now let's go for llm and web searxch

In [62]:
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")

client = Groq()


def generate_response(
    prompt: str,
    model: str = "openai/gpt-oss-120b",
    max_tokens: int = 200,
    system_prompt: str = "You are a helpful AI assistant."
):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=max_tokens,
    )

    return {
        "content": response.choices[0].message.content,
        "usage": response.usage
    }


# Example usage
result = generate_response(
    prompt="Explain an AI agent in 2 sentences."
)

print(result["content"])
print(result["usage"])

An AI agent is a software entity that perceives its environment through data inputs, processes that information using algorithms or learned models, and then takes actions to achieve specific goals. By continuously observing outcomes and adapting its behavior, the agent can improve performance over time, much like an autonomous decision‑making system.
CompletionUsage(completion_tokens=93, prompt_tokens=90, total_tokens=183, completion_time=0.193724917, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=24), prompt_time=0.003643834, prompt_tokens_details=None, queue_time=0.284088015, total_time=0.197368751)


In [63]:
import os, requests
from dotenv import load_dotenv
load_dotenv("../.env")

def web_search(query, num=5):
    r = requests.get(
        "https://serpapi.com/search",
        params={
            "engine": "google",
            "q": query,
            "num": num,
            "api_key": os.getenv("SERPAPI_API_KEY"),
        },
        timeout=30,
    )
    print("Status:", r.status_code)   # keep this while debugging
    r.raise_for_status()
    return r.json().get("organic_results", [])

for item in web_search("what is agentic AI"):
    print(item["title"], "|", item["link"])
    print(item.get("snippet", ""), "\n")


Status: 200
What is Agentic AI? | https://www.ibm.com/think/topics/agentic-ai
Agentic AI is an artificial intelligence system that can accomplish a specific goal with limited supervision. It consists of ai agents—machine learning ... 

Agentic AI, explained | https://mitsloan.mit.edu/ideas-made-to-matter/agentic-ai-explained
He defines agentic AI as systems that incorporate multiple, different agents that are orchestrating a task together — for example, a marketplace ... 

What is agentic AI? Definition and differentiators | https://cloud.google.com/discover/what-is-agentic-ai
Agentic AI is an advanced form of artificial intelligence focused on autonomous decision-making and action. Unlike traditional AI, which ... 

What is Agentic AI? - AWS - Amazon.com | https://www.google.com/goto?url=CAESZwHEIkXXA4OK3TSZ1Re9ABDjXpC2e_Aq22B_C8ueJ_qL5s_C0A1Os6SIlrlUr_9hn80HbkZxmcbv_ZcqzzxjgM-lcyA1mNMabijM5SEHlj2YY8CNMPpSTUXRZDsAr8jchE2-VqfILa4
Agentic AI is an autonomous AI system that can act indep